In [1]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import gc

torch.set_num_threads(1)

import sys
sys.path.append("..")
from src.pinn import BlackScholesPINN

In [4]:
# Defining parameters.

K = 100  # Strike price
T = 1.0  # Time to maturity
r = 0.05  # Risk-free interest rate
sigma = 0.2  # Volatility
S_max = 3*K # Maximum stock price considered

In [3]:
# Generating training points for the PINN model.

def generate_training_points(N_f=500, N_ic=200, N_bc=200):
    S_f = torch.rand(N_f, 1) * S_max
    tau_f = torch.rand(N_f, 1) * T

    S_ic = torch.rand(N_ic, 1) * S_max
    tau_ic = torch.zeros_like(S_ic)
    V_ic = torch.clamp(S_ic - K, min=0.0)

    tau_lb = torch.rand(N_bc, 1) * T
    S_lb = torch.zeros_like(tau_lb)
    V_lb = torch.zeros_like(tau_lb)

    tau_ub = torch.rand(N_bc, 1) * T
    S_ub = torch.full_like(tau_ub, S_max)
    V_ub = S_max - K * torch.exp(-r * tau_ub)

    return S_f, tau_f, S_ic, tau_ic, V_ic, S_lb, tau_lb, V_lb, S_ub, tau_ub, V_ub

In [5]:
# Defining the loss function for the PINN model.

def pinn_loss(model, training_data):
    S_f, tau_f, S_ic, tau_ic, V_ic, S_lb, tau_lb, V_lb, S_ub, tau_ub, V_ub = training_data

    residual = model.pde_residual(S_f, tau_f, r, sigma)
    loss_pde = torch.mean((residual / K) ** 2)

    V_ic_pred = model(S_ic, tau_ic)
    loss_ic = torch.mean(((V_ic_pred - V_ic) / K) ** 2)

    V_lb_pred = model(S_lb, tau_lb)
    loss_lb = torch.mean(((V_lb_pred - V_lb) / K) ** 2)

    V_ub_pred = model(S_ub, tau_ub)
    loss_ub = torch.mean(((V_ub_pred - V_ub) / K) ** 2)

    total_loss = loss_pde + loss_ic + loss_lb + loss_ub

    return total_loss, loss_pde, loss_ic, loss_lb, loss_ub

In [6]:
# Training the PINN model.

model = BlackScholesPINN(
    layers=[2, 32, 32, 32, 1],
    S_max=S_max,
    T=T,
    price_scale=K
)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

loss_history = []

epochs = 2000


def train_step():
    training_data = generate_training_points(N_f=200, N_ic=100, N_bc=100)

    optimizer.zero_grad(set_to_none=True)

    total_loss, loss_pde, loss_ic, loss_lb, loss_ub = pinn_loss(model, training_data)

    total_loss.backward()
    optimizer.step()

    values = (
        total_loss.detach().item(),
        loss_pde.detach().item(),
        loss_ic.detach().item(),
        loss_lb.detach().item(),
        loss_ub.detach().item()
    )

    del training_data
    del total_loss, loss_pde, loss_ic, loss_lb, loss_ub

    return values


for epoch in range(epochs):
    total, pde, ic, lb, ub = train_step()

    loss_history.append(total)

    if epoch % 100 == 0:
        gc.collect()

    if epoch % 500 == 0:
        print(
            f"Epoch {epoch}: "
            f"Total={total:.6e}, "
            f"PDE={pde:.6e}, "
            f"IC={ic:.6e}, "
            f"LB={lb:.6e}, "
            f"UB={ub:.6e}"
        )

Epoch 0: Total=5.056012e+00, PDE=2.518194e-03, IC=9.254209e-01, LB=4.072551e-03, UB=4.124000e+00
Epoch 500: Total=2.395402e-02, PDE=3.914942e-03, IC=1.490842e-02, LB=4.225381e-03, UB=9.052809e-04
Epoch 1000: Total=5.785742e-03, PDE=1.845991e-03, IC=3.353318e-03, LB=3.373604e-04, UB=2.490723e-04
Epoch 1500: Total=3.176946e-03, PDE=5.968211e-04, IC=2.327967e-03, LB=8.694470e-05, UB=1.652133e-04


In [7]:
# Visualizing the training loss history.

plt.figure(figsize=(8, 5))
plt.plot(loss_history)
plt.yscale("log")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("PINN Training Loss")
plt.grid(True)
plt.show()

: 